In [ ]:
import os
import requests
import pandas as pd
import logging
from dotenv import load_dotenv
from datetime import datetime, timedelta
from time import sleep, time as now

# === LOGGING ===
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# === CONFIG ===
force_clean_start = True
output_dir = r"C:\Android Mobile App\Step 1_URL_Search_Kotlin"
output_csv_raw = os.path.join(output_dir, "github_android_search_results_raw.csv")
output_csv_filtered = os.path.join(output_dir, "github_android_search_results_filtered.csv")
output_ranges_csv = os.path.join(output_dir, "final_ranges_used.csv")

os.makedirs(output_dir, exist_ok=True)

# === AUTH ===
load_dotenv("All_Tokens.env")
token = os.getenv("GITHUB_TOKEN_1") or os.getenv("GITHUB_TOKEN")
if not token:
    raise ValueError("❌ No GitHub token found in All_Tokens.env")

HEADERS = {
    "Authorization": f"token {token}",
    "Accept": "application/vnd.github+json"
}

# === SETTINGS ===
start_date = datetime.strptime("2008-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2024-12-31", "%Y-%m-%d")
initial_window_hours = 15 * 24
min_window_hours = 1
max_window_hours = 90 * 24
MAX_RESULTS_PER_QUERY = 1000
TARGET_FILL_RATIO = 0.25

# === SINGLE BASE QUERY ===
base_query_prefix = "stars:>0 language:Kotlin fork:false archived:false"

def filter_item(item):
    return (
        item.get('stargazers_count', 0) > 0
        and not item.get('fork', False)
        and not item.get('archived', False)
        and (item.get('language') or '').lower() == 'kotlin'
    )

def check_rate_limit():
    r = requests.get("https://api.github.com/rate_limit", headers=HEADERS)
    if r.status_code != 200:
        logger.warning("⚠️ Could not check rate limit.")
        return
    data = r.json()
    remaining = data['resources']['search']['remaining']
    reset_epoch = data['resources']['search']['reset']
    reset_in = max(0, reset_epoch - now())
    logger.info(f"🔎 Search API remaining: {remaining} | resets in {reset_in/60:.1f} min")
    if remaining < 5:
        logger.warning(f"⏳ Low quota → Sleeping for {reset_in/60:.1f} min.")
        sleep(reset_in + 5)

def check_count(query):
    while True:
        check_rate_limit()
        r = requests.get("https://api.github.com/search/repositories", headers=HEADERS, params={"q": query, "per_page": 1})
        if r.status_code == 200:
            return r.json().get("total_count", 0)
        else:
            logger.warning(f"Retrying count check: {r.status_code}")
            sleep(5)

def fetch_items(query):
    all_items = []
    for page in range(1, 11):
        check_rate_limit()
        r = requests.get("https://api.github.com/search/repositories", headers=HEADERS, params={"q": query, "per_page": 100, "page": page})
        if r.status_code == 200:
            items = r.json().get("items", [])
            if not items:
                break
            all_items.extend(items)
            sleep(1)
        else:
            logger.warning(f"Retrying fetch page {page}: {r.status_code}")
            sleep(5)
    return all_items

# === SMART RESUME ===
if os.path.exists(output_ranges_csv):
    df_ranges = pd.read_csv(output_ranges_csv)
    if not df_ranges.empty:
        last_end = df_ranges['end_date'].iloc[-1]
        current_start = datetime.fromisoformat(last_end) + timedelta(seconds=1)
    else:
        current_start = start_date
else:
    current_start = start_date

if force_clean_start:
    for f in [output_csv_raw, output_csv_filtered, output_ranges_csv]:
        if os.path.exists(f):
            os.remove(f)
            logger.info(f"🧹 Removed old file: {f}")
    current_start = start_date

all_results = []
final_ranges = []

if os.path.exists(output_csv_raw):
    all_results = pd.read_csv(output_csv_raw).to_dict('records')

if os.path.exists(output_ranges_csv):
    final_ranges = pd.read_csv(output_ranges_csv).to_dict('records')

current_window_hours = initial_window_hours

while current_start < end_date:
    window_hours = current_window_hours
    while True:
        current_end = min(current_start + timedelta(hours=window_hours), end_date)
        date_range = f"created:{current_start.isoformat()}..{current_end.isoformat()}"
        base_query = f"{base_query_prefix} {date_range}"

        total_count = check_count(base_query)
        logger.info(f"⏳ Checking: {base_query} → {total_count} repos")

        if total_count >= MAX_RESULTS_PER_QUERY and window_hours > min_window_hours:
            window_hours = max(window_hours // 2, min_window_hours)
            continue
        elif total_count < MAX_RESULTS_PER_QUERY * TARGET_FILL_RATIO and window_hours * 2 <= max_window_hours:
            window_hours = min(window_hours * 2, max_window_hours)

        items = [item for item in fetch_items(base_query) if filter_item(item)]
        for item in items:
            item['search_qualifier'] = base_query
            item['repo_stars'] = item.get('stargazers_count', 0)

        all_results.extend(items)
        final_ranges.append({
            "start_date": current_start.isoformat(),
            "end_date": current_end.isoformat(),
            "result_count": total_count,
            "window_hours": window_hours
        })

        pd.json_normalize(all_results).to_csv(output_csv_raw, index=False)
        pd.DataFrame(final_ranges).to_csv(output_ranges_csv, index=False)

        logger.info(f"💾 Progress saved: {len(all_results)} repos")
        sleep(2)
        break

    current_start = current_end + timedelta(seconds=1)
    current_window_hours = window_hours

# === FINAL FILTER ===
df = pd.json_normalize(all_results)
keep_fields = [
    'language', 'search_qualifier', 'name', 'full_name', 'private', 'html_url', 'url',
    'clone_url', 'visibility', 'owner.login', 'size', 'stargazers_count',
    'watchers_count', 'forks', 'open_issues', 'default_branch',
    'open_issues_count', 'repo_stars', 'topics', 'description', 'fork', 'archived'
]
for f in keep_fields:
    if f not in df.columns:
        df[f] = None

df = df[keep_fields]
df.to_csv(output_csv_filtered, index=False)
logger.info(f"✅ Filtered fields saved to: {output_csv_filtered}")
